# Sentinel-2 field time series: `nl_2025` on Colab

Runs the team's frozen extractor (fingerprint `9cd0447b`) with the same settings as the
laptop. Order: upload the files listed in `README.md` into `/content`, then run the cells top to bottom.
Step 4 re-extracts two blocks the laptop already finished and **stops the notebook if a single value differs**.
To resume after a restart, run all cells again: finished blocks are skipped.

In [ ]:
CHECK_SCRIPT = '\n"""Same-result check: re-extract finished laptop blocks and compare exactly."""\nimport json, shutil, sys, zipfile\nfrom argparse import Namespace\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom fieldvision.s2.core import BANDS\nfrom fieldvision.s2.extract import process_block\nfrom fieldvision.s2.geometry import load_fields, make_blocks\n\nwork, check_zip, workers = Path(sys.argv[1]), Path(sys.argv[2]), int(sys.argv[3])\nblock_ids = sys.argv[4:]\nroot = work / "check"\nshutil.rmtree(root, ignore_errors=True)\nzipfile.ZipFile(check_zip).extractall(work)\nEXACT = ["scene_id", "collection", "n_pixels", "n_clear", "clear_fraction", *BANDS]\nfailed = False\nwith ThreadPoolExecutor(max_workers=workers) as pool:\n    for block_id in block_ids:\n        d = root / block_id\n        s = json.loads((d / "settings.json").read_text())\n        p = s["parameters"]\n        blocks = {b["id"]: b for b in make_blocks(load_fields(d / "fields.geojson", s["year"]), p["block_size"])}\n        block = blocks[block_id]\n        out = d / "out"\n        (out / "blocks").mkdir(parents=True)\n        (out / "catalog").mkdir()\n        shutil.copy(d / "catalog.json", out / "catalog" / f"{block_id}.json")\n        # Transient HTTP read failures are retried (the extractor resumes failed scenes);\n        # only a block that reads completely is compared.\n        for attempt in range(4):\n            status = process_block(block, Namespace(**p, out=out, scene_batch=6), pool)\n            errors = status.get("scene_errors", {})\n            if status["status"] == "complete" or not all("RasterioIOError" in e or "CPLE_" in e for e in errors.values()):\n                break\n            print(f"{block_id}: {len(errors)} scene reads failed (network), retry {attempt + 1}", flush=True)\n            import time\n            time.sleep(30 * (attempt + 1))\n        old = pd.read_parquet(d / "reference.parquet").set_index(["field_id", "date"]).sort_index()\n        new = pd.read_parquet(out / "blocks" / f"{block_id}.parquet").set_index(["field_id", "date"]).sort_index()\n        problems = []\n        if status["status"] != "complete":\n            problems.append(f"block status {status[\'status\']}: {status.get(\'scene_errors\')}")\n        if not old.index.equals(new.index):\n            problems.append(f"rows differ: laptop {len(old)}, here {len(new)}, "\n                            f"{len(old.index.symmetric_difference(new.index))} keys not shared")\n            common = old.index.intersection(new.index)\n            old, new = old.loc[common], new.loc[common]\n        for column in EXACT:\n            a, b = old[column].to_numpy(), new[column].to_numpy()\n            same = (a == b) | (pd.isna(a) & pd.isna(b)) if a.dtype.kind == "f" else (a == b)\n            if not np.all(same):\n                first = np.flatnonzero(~same)[0]\n                key = old.index[first]\n                problems.append(f"{column}: {int((~same).sum())} rows differ, e.g. {key}: laptop {a[first]!r}, here {b[first]!r}")\n        print(f"{block_id} ({s[\'source_run\']}): {len(old)} rows, {status[\'scenes_succeeded\']} scenes -> "\n              + ("IDENTICAL" if not problems else "DIFFERENT"))\n        for problem in problems:\n            print("   ", problem)\n        failed |= bool(problems)\nprint("CHECK", "FAILED" if failed else "PASSED")\nsys.exit(1 if failed else 0)\n'
# 1. Settings (only WORKERS may be changed; it does not change results)
import os, sys, json, time, shutil, zipfile, hashlib, subprocess, threading
from pathlib import Path
UPLOAD = Path("/content")
WORK = Path("/content/fv")
bundle = json.loads((UPLOAD / "bundle.json").read_text())
RUN, YEAR = bundle["run"], bundle["year"]
OUT = WORK / RUN
CODE = WORK / "code"
DRIVE_BACKUP = Path("/content/drive/MyDrive/fieldvision_" + RUN)
CPUS = os.cpu_count() or 2
WORKERS = max(16, 8 * CPUS)   # I/O threads for (scene, band) reads
print(f"{RUN} {YEAR}: {CPUS} CPUs, {WORKERS} read threads; Python {sys.version.split()[0]}")
missing = [n for n in [bundle["code_zip"], bundle["fields"], "check_blocks.zip", *bundle["catalog_parts"]]
           if not (UPLOAD / n).exists()]
print("missing uploads:", missing or "none")

In [ ]:
# 2. Optional, recommended: mount Google Drive so checkpoints survive a recycled runtime
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3. Install the exact package versions used on the laptop (binary wheels only) and print them
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", *"numpy==2.5.4 pandas==3.1.1 pyarrow==26.0.1 rasterio==1.5.2 shapely==2.2.0 pyproj==3.8.0 requests==2.34.2 urllib3==2.8.0 affine==3.0.1".split()], check=True)
versions = subprocess.run([sys.executable, "-c", (
    "import importlib.metadata as m, rasterio, pyproj, sys;"
    "print('python', sys.version.split()[0]);"
    "[print(p, m.version(p)) for p in ['numpy', 'pandas', 'pyarrow', 'rasterio', 'shapely', 'pyproj', 'requests', 'urllib3', 'affine']];"
    "print('GDAL', rasterio.__gdal_version__); print('PROJ', pyproj.proj_version_str)")],
    capture_output=True, text=True, check=True).stdout
print(versions)
print("laptop:", bundle["laptop"], "(different GDAL/PROJ builds are allowed only if step 4 passes)")

In [ ]:
# 4. Unpack the frozen code, check its fingerprint, then the same-result check
shutil.rmtree(CODE, ignore_errors=True)
zipfile.ZipFile(UPLOAD / bundle["code_zip"]).extractall(CODE)
digest = hashlib.sha256(b"".join(p.read_bytes() for p in sorted((CODE / "fieldvision" / "s2").glob("*.py")))).hexdigest()
print("code fingerprint", digest)
assert digest == bundle["code_sha256"], "code fingerprint differs from the laptop's frozen copy"
WORK.mkdir(parents=True, exist_ok=True)
(WORK / "check.py").write_text(CHECK_SCRIPT)
env = dict(os.environ, PYTHONPATH=str(CODE))
passed_marker = WORK / "check_passed.txt"
stamp = digest + "\n" + versions
if passed_marker.exists() and passed_marker.read_text() == stamp:
    print("same-result check already PASSED in this runtime with this code and these libraries; not repeated")
else:
    result = subprocess.run([sys.executable, "-W", "ignore", str(WORK / "check.py"), str(WORK), str(UPLOAD / "check_blocks.zip"),
                             str(min(WORKERS, 16)), *bundle["check_blocks"]], env=env, capture_output=True, text=True)
    print(result.stdout[-6000:])
    if result.returncode != 0:
        print(result.stderr[-3000:])
        raise SystemExit("Same-result check FAILED: do not use this runtime for the extraction. Tell the coordinator.")
    passed_marker.write_text(stamp)

In [ ]:
# 5. Prepare the output folder: restore from Drive if this is a fresh runtime, add the catalog cache
OUT.mkdir(parents=True, exist_ok=True)
if DRIVE_BACKUP.exists() and not (OUT / "manifest.json").exists():
    shutil.copytree(DRIVE_BACKUP, OUT, dirs_exist_ok=True)
    print("restored checkpoints from Drive:", len(list((OUT / "blocks").glob("*.json"))), "block records")
parts = b"".join((UPLOAD / n).read_bytes() for n in bundle["catalog_parts"])
(WORK / "catalog.zip").write_bytes(parts)
added = 0
with zipfile.ZipFile(WORK / "catalog.zip") as archive:
    for name in archive.namelist():
        target = OUT / name
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(name))
            added += 1
print("catalog files added:", added, "| blocks already finished:",
      sum(json.loads(p.read_text()).get("status") in ("complete", "partial") for p in (OUT / "blocks").glob("*.json"))
      if (OUT / "blocks").exists() else 0)

In [ ]:
# 6. Run the extraction (same command and settings as the laptop lane). Rerun this cell to resume.
cmd = [sys.executable, "-W", "ignore", "-m", "fieldvision.s2.extract", "--fields", str(UPLOAD / bundle["fields"]),
       "--year", str(YEAR), "--start", bundle["start"], "--end", bundle["end"], "--out", str(OUT),
       "--workers", str(WORKERS), "--scene-batch", str(bundle["scene_batch"])]
print(" ".join(cmd))
total_blocks = len(list((OUT / "catalog").glob("*.json")))
stop = threading.Event()

def backup():
    while not stop.wait(300):
        if Path("/content/drive/MyDrive").exists():
            for item in ["manifest.json", "blocks"]:
                src = OUT / item
                if src.is_dir():
                    shutil.copytree(src, DRIVE_BACKUP / item, dirs_exist_ok=True,
                                    ignore=shutil.ignore_patterns("*.tmp"))
                elif src.exists():
                    DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(src, DRIVE_BACKUP / item)

threading.Thread(target=backup, daemon=True).start()
started = time.time()
with open(OUT / "run.log", "a") as log:
    proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        log.write(line)
        if "boto3" in line:
            continue
        if " fields, " in line or "WARNING" in line or "ERROR" in line or ": complete" in line or "partial:" in line or "complete:" in line:
            done = sum(json.loads(p.read_text()).get("status") in ("complete", "partial") for p in (OUT / "blocks").glob("*.json"))
            print(f"[{time.strftime('%H:%M:%S')}] {done}/{total_blocks} blocks done | {line.strip()[24:]}", flush=True)
    code_ = proc.wait()
stop.set()
print("exit code", code_, "(0 complete, 2 partial: see scene errors in manifest)", f"{(time.time()-started)/60:.1f} min this session")
failures = {}
for p in (OUT / "blocks").glob("*.json"):
    for e in json.loads(p.read_text()).get("scene_errors", {}).values():
        kind = "network read (rerun this cell)" if ("RasterioIOError" in e or "CPLE_" in e) else e[:60]
        failures[kind] = failures.get(kind, 0) + 1
print("scene errors by kind:", failures or "none")

In [ ]:
# 7. Results: counts, then one zip with the same outputs as a laptop run (catalog left out)
import pandas as pd
manifest = json.loads((OUT / "manifest.json").read_text())
coverage = pd.read_csv(OUT / "fields_coverage.csv")
table = pd.read_parquet(OUT / "timeseries.parquet")
usable = table[(table.n_clear >= 1) & (table.clear_fraction >= 0.2)]
print("status", manifest["status"], "| rows", len(table), "| usable rows", len(usable), "| fields", len(coverage),
      "| field status", coverage.status.value_counts().to_dict())
print("usable dates per field min/median/max", coverage.usable_dates.min(), coverage.usable_dates.median(), coverage.usable_dates.max())
print("rows by source", table.collection.value_counts().to_dict(), "| scene failures", manifest["scene_failures"])
result_zip = Path(f"/content/{RUN}_colab_result.zip")
with zipfile.ZipFile(result_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT.rglob("*")):
        if path.is_file() and "catalog" not in path.relative_to(OUT).parts and not path.name.endswith(".tmp"):
            archive.write(path, path.relative_to(OUT).as_posix())
    archive.writestr("colab_environment.txt", versions + f"\ncode fingerprint {digest}\nsame-result check PASSED\n")
print(result_zip, f"{result_zip.stat().st_size/1e6:.1f} MB")
if Path("/content/drive/MyDrive").exists():
    shutil.copy2(result_zip, Path("/content/drive/MyDrive") / result_zip.name)
    print("copied to Drive:", result_zip.name)
from google.colab import files
files.download(str(result_zip))